# Перекрёстная проверка, группы и время

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Выбирать схему проверки, повторяющую появление новых объектов, групп или будущих наблюдений.

## Одного разбиения бывает мало

При перекрёстной проверке обучающую часть делят на K блоков. По-английски один такой блок называют fold, а всю схему — K-fold. По очереди один блок оставляют для проверки, а на остальных обучают модель и все преобразования признаков заново. Затем усредняют оценки и сохраняют их разброс. Итоговый тест в этой процедуре не участвует. Здесь K — число блоков проверки; это другая настройка, чем k соседей в KNN. Имена `valid` и `validation` в примерах обозначают проверочную часть для выбора; финальный `test` сохраняет отдельную роль.

**Стратификация** старается сохранить доли классов в блоках. Она помогает избежать блока без редкого класса, но не делает зависимые объекты независимыми. Если от одного ученика есть десять снимков, случайное разбиение снимков проверяет знакомого ученика, а не нового.

## Разобранный пример: четыре ученика

Есть по два измерения учеников А,Б,В,Г. Для проверки на новых учениках обе строки А должны оказаться в одной части. В GroupKFold параметр groups содержит идентификатор ученика; этот столбец используется разбиением и не передаётся модели как числовой признак.

Если задача прогнозирует будущие дни, проверка должна идти вперёд по времени: обучаться на прошлом и проверяться на следующем отрезке. Перемешивание позволяет будущим закономерностям попасть в обучение и отвечает на другой вопрос. Иногда нужен зазор между частями, если близкие по времени объекты используют перекрывающиеся окна.

Число блоков ограничено объёмом данных и ценой обучения. При слишком редком классе нельзя потребовать, чтобы он присутствовал в каждом из десяти блоков, если положительных объектов всего три. Полезнее честно описать ограничение, чем формально вызвать разбиение с неподходящими параметрами.

`np.repeat(values, 2)` повторяет каждый элемент дважды, создавая пары идентификаторов. `GroupKFold.split` возвращает индексы train и validation с учётом `groups`; `TimeSeriesSplit` создаёт последовательные временные части. Метод множества `isdisjoint` проверяет пустое пересечение.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold, TimeSeriesSplit
X = np.arange(8).reshape(-1, 1)
groups = np.repeat(["a", "b", "c", "d"], 2)
for train, valid in GroupKFold(n_splits=2).split(X, groups=groups):
    assert set(groups[train]).isdisjoint(set(groups[valid]))
for train, valid in TimeSeriesSplit(n_splits=2).split(X):
    assert max(train) < min(valid)
print("Группы разделены; время направлено вперёд")


## Типичные ошибки

Не передавайте идентификатор группы как полезный признак без обоснования. Не обучайте подготовку заранее на всех блоках.

## Задачи

### 1. Базовая

Что называют блоком (fold) перекрёстной проверки?

<details>
<summary>Подсказка</summary>

Часть для одного шага проверки.

</details>



### 2. Базовая

Используют ли итоговый тест при подборе настроек с перекрёстной проверкой?

<details>
<summary>Подсказка</summary>

Он должен остаться закрытым.

</details>



### 3. Прикладная

При перекрёстной проверке по пяти блокам сколько раз каждый объект попадает в проверочную часть?

<details>
<summary>Подсказка</summary>

Каждый блок оставляют один раз.

</details>



### 4. Прикладная

Две фотографии одного человека: куда их поместить для проверки новых людей?

<details>
<summary>Подсказка</summary>

Единица независимости — человек.

</details>



### 5. Олимпиадная

Гарантирует ли стратификация отсутствие утечки?

<details>
<summary>Подсказка</summary>

Она контролирует только доли классов.

</details>



### 6. Олимпиадная

Почему усреднение по блокам не отменяет зависимость наблюдений?

<details>
<summary>Подсказка</summary>

Проблема содержится в сборе и разделении объектов.

</details>



## Итоги и самопроверка

Выберите схему для новых пользователей и для следующей недели. Объясните, почему одна случайная стратификация не решает обе задачи.

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Один блок разбиения; он по очереди служит внутренней проверкой.

### Задача 2

Нет. Подбор использует только данные разработки.

### Задача 3

Один раз, если провести один полный цикл проверки по всем блокам.

### Задача 4

В одну и ту же часть по идентификатору группы.

### Задача 5

Нет. Повторы людей, будущие данные и признаки по ответу требуют отдельных проверок.

### Задача 6

Повторное использование зависимых строк сохраняет лёгкую проверку. Нужна подходящая единица разбиения.